# BiomedGPT CXR Fine-tuning — Fix Mode Collapse

`iu-xray-finetuned` collapses onto ~13 canned "normal" templates for ~60% of images.
This notebook re-trains from `instruct-biomedgpt-base` with a rebalanced sampler that
oversamples reports with a real finding, so it can't win by always predicting "normal."

**Runtime:** GPU. **Data:** `dz-osamu/IU-Xray` (2,069 train / 296 val / 590 test).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only -- check Runtime > Change runtime type")


## 1. Environment setup
Same approach validated in the eval notebook: shadow Colab's stock `transformers` with the OFA-Sys fork via `sys.path`, no `pip install -e` (avoids the restart-required issue and the `sentence-transformers` dependency conflict).

In [ ]:
import os

if not os.path.isdir("OFA"):
    !git clone --single-branch --branch feature/add_transformers https://github.com/OFA-Sys/OFA.git
else:
    print("OFA/ already exists, skipping clone.")

!pip install -q pillow torchvision huggingface_hub datasets sacrebleu rouge-score sacremoses


### `huggingface_hub` compatibility shim

This fork imports some APIs (`HfFolder`, `Repository`) that newer `huggingface_hub`
removed, which crashes `import transformers` unless patched first. We monkey-patch the
`huggingface_hub` module directly rather than editing every file that imports them.

In [ ]:
import huggingface_hub

class _RemovedHubAPI:
    def __getattr__(self, name):
        raise RuntimeError("This deprecated huggingface_hub API was removed upstream and is not used by this project.")
    def __call__(self, *a, **kw):
        raise RuntimeError("This deprecated huggingface_hub API was removed upstream and is not used by this project.")

_shim = _RemovedHubAPI()
patched = []
for _name in ["HfFolder", "Repository", "create_repo", "list_repo_files", "whoami"]:
    if not hasattr(huggingface_hub, _name):
        setattr(huggingface_hub, _name, _shim)
        patched.append(_name)
print("shimmed:", patched if patched else "(none needed, all present)")


In [ ]:
import sys

sys.path.insert(0, os.path.abspath("OFA/transformers/src"))
import transformers
print("transformers loaded from:", transformers.__file__)
assert "OFA/transformers/src" in transformers.__file__, "Stock transformers shadowed our fork -- rerun this cell, or restart runtime if it persists."


In [ ]:
import re

path = "OFA/transformers/src/transformers/models/ofa/modeling_ofa.py"
with open(path) as f:
    src = f.read()

old = 'image_position_idx = torch.cat([image_position_idx, torch.tensor([1024] * 768)])'
new = 'image_position_idx = torch.cat([image_position_idx, torch.tensor([1024] * 769)])'

if new in src:
    print("Already patched, skipping.")
elif old in src:
    src = src.replace(old, new)
    with open(path, "w") as f:
        f.write(src)
    print("Patched.")
else:
    raise AssertionError("Neither old nor new line found -- fork may have changed, inspect manually")


## 2. Download the base checkpoint

Starting from the generic instruct-tuned checkpoint, not the already-collapsed IU-X-ray fine-tune.

In [ ]:
from huggingface_hub import snapshot_download

base_dir = snapshot_download(repo_id="PanaceaAI/instruct-biomedgpt-base", local_dir="checkpoints/instruct-biomedgpt-base")
print("Downloaded to", base_dir)


## 3. Download and prepare the IU X-ray train/val data

Images live in a separate `image.zip`, whose internal paths don't match the paths
recorded in `train.jsonl`/`val.jsonl`. We match on study folder + filename instead of
the full path.

In [ ]:
import os

os.makedirs("iu_xray_data", exist_ok=True)
for fname in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    dest = f"iu_xray_data/{fname}"
    if not os.path.exists(dest):
        !curl -L -s "https://huggingface.co/datasets/dz-osamu/IU-Xray/resolve/main/{fname}" -o {dest}
    print(fname, os.path.getsize(dest), "bytes")

zip_path = "iu_xray_data/image.zip"
if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 1_000_000_000:
    !curl -L "https://huggingface.co/datasets/dz-osamu/IU-Xray/resolve/main/image.zip" -o {zip_path} --progress-bar
else:
    print("image.zip already downloaded, skipping.")


In [ ]:
import zipfile

extract_dir = "iu_xray_data/images"
if not os.path.isdir(extract_dir) or not os.listdir(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print("Extracted.")
else:
    print("Already extracted, skipping.")

# Build an index: "STUDY_FOLDER/N.png" -> actual extracted file path
image_index = {}
for root, dirs, files in os.walk(extract_dir):
    for fname in files:
        if fname.endswith(".png"):
            key = os.path.join(os.path.basename(root), fname)
            image_index[key] = os.path.join(root, fname)

print("Indexed", len(image_index), "images")
sample_key = next(iter(image_index))
print("Example key:", sample_key, "->", image_index[sample_key])


In [ ]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def resolve_image_path(dataset_path):
    # dataset_path looks like "/iu_xray/image/CXR1000_IM-0003/0.png" -- match on last 2 segments
    parts = dataset_path.strip("/").split("/")
    key = os.path.join(parts[-2], parts[-1])
    return image_index.get(key)

train_records = load_jsonl("iu_xray_data/train.jsonl")
val_records = load_jsonl("iu_xray_data/val.jsonl")
print("train:", len(train_records), "val:", len(val_records))
print(train_records[0].keys())

# sanity check: how many records resolve to a real file?
unresolved = 0
for r in train_records[:100]:
    if resolve_image_path(r["images"][0]) is None:
        unresolved += 1
print(f"unresolved in first 100 train records: {unresolved}/100")


## 4. Rebalanced sampler

Flags each report as "has a real finding" vs "normal-template-like" using the same
negation-aware keyword detector built for evaluation, then oversamples the findings
cases so the model can't win by always predicting the majority template.

In [ ]:
import re as re_module

NEGATION_CUES = ["no ", "without ", "not ", "free of ", "negative for ", "absence of ", "resolution of ", "resolved "]
NEGATION_WINDOW_CHARS = 40
FINDING_TERMS = [
    "cardiomegaly", "enlarged heart", "opacity", "opacities", "opacification", "lesion", "nodule", "mass",
    "edema", "consolidation", "pneumonia", "atelectasis", "pneumothorax", "effusion", "pleural thickening",
    "fracture", "hernia", "emphysema", "scarring",
]

def has_real_finding(text):
    text = text.lower()
    for term in FINDING_TERMS:
        for m in re_module.finditer(re_module.escape(term), text):
            window = text[max(0, m.start() - NEGATION_WINDOW_CHARS):m.start()]
            if not any(c in window for c in NEGATION_CUES):
                return True
    return False

finding_flags = [has_real_finding(r["response"]) for r in train_records]
n_finding = sum(finding_flags)
print(f"{n_finding}/{len(train_records)} train reports flagged as having a real finding ({n_finding/len(train_records):.1%})")

# oversample findings cases to roughly balance the two groups
weight_finding = 1.0
weight_normal = n_finding / (len(train_records) - n_finding) if (len(train_records) - n_finding) > 0 else 1.0
sample_weights = [weight_finding if f else weight_normal for f in finding_flags]
print(f"weight for normal-template reports: {weight_normal:.3f} (findings reports weighted 1.0)")


## 5. Dataset and training loop

Uses the same prompt (`" what does the image describe?"`) as the rest of this project's inference pipeline, so the fine-tuned model stays a drop-in replacement -- training on a different instruction template would create a train/inference mismatch.

In [ ]:
from PIL import Image
from torchvision import transforms
from transformers import OFATokenizer, OFAModel
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

CAPTION_PROMPT = " what does the image describe?"
IMAGE_RESOLUTION = 256
IMAGE_MEAN = IMAGE_STD = [0.5, 0.5, 0.5]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = OFATokenizer.from_pretrained("checkpoints/instruct-biomedgpt-base")

transform = transforms.Compose([
    lambda image: image.convert("RGB"),
    transforms.Resize((IMAGE_RESOLUTION, IMAGE_RESOLUTION), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

PROMPT_IDS = tokenizer([CAPTION_PROMPT], return_tensors="pt").input_ids[0]  # shared across all examples


class IUXrayDataset(Dataset):
    def __init__(self, records, max_target_len=64):
        self.records = [r for r in records if resolve_image_path(r["images"][0]) is not None]
        print(f"Dataset: {len(self.records)}/{len(records)} records have a resolvable image")
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        image = Image.open(resolve_image_path(r["images"][0]))
        patch_image = transform(image)
        target_ids = tokenizer(
            r["response"], return_tensors="pt", max_length=self.max_target_len, truncation=True
        ).input_ids[0]
        return patch_image, target_ids


def collate_fn(batch):
    # each b[1] is already tokenizer output: [bos, c1, ..., cn, eos]
    patch_images = torch.stack([b[0] for b in batch])
    full_ids_list = [b[1] for b in batch]
    max_len = max(len(t) for t in full_ids_list)
    pad_id = tokenizer.pad_token_id

    full_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, t in enumerate(full_ids_list):
        full_ids[i, :len(t)] = t

    # teacher forcing via slicing the SAME padded tensor two ways:
    # decoder_input_ids = [bos, c1, ..., cn]  (drop trailing eos/pad column)
    # targets           = [c1, ..., cn, eos]  (drop leading bos column)
    # correct for any padding length because loss ignores pad_id positions in targets.
    decoder_input_ids = full_ids[:, :-1]
    targets = full_ids[:, 1:]

    prompt_ids = PROMPT_IDS.unsqueeze(0).expand(len(batch), -1)
    return prompt_ids, patch_images, decoder_input_ids, targets


train_dataset = IUXrayDataset(train_records)
val_dataset = IUXrayDataset(val_records)

# re-filter sample_weights to match records actually kept by the dataset (resolvable images)
kept_indices = [i for i, r in enumerate(train_records) if resolve_image_path(r["images"][0]) is not None]
kept_weights = [sample_weights[i] for i in kept_indices]
sampler = WeightedRandomSampler(kept_weights, num_samples=len(kept_weights), replacement=True)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"train batches: {len(train_loader)}, val batches: {len(val_loader)}")


In [ ]:
model = OFAModel.from_pretrained("checkpoints/instruct-biomedgpt-base", use_cache=False).to(DEVICE)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
label_smoothing = 0.1
pad_id = tokenizer.pad_token_id
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=label_smoothing)


def run_forward(prompt_ids, patch_images, decoder_input_ids, targets):
    prompt_ids = prompt_ids.to(DEVICE)
    patch_images = patch_images.to(DEVICE)
    decoder_input_ids = decoder_input_ids.to(DEVICE)
    targets = targets.to(DEVICE)

    output = model(
        input_ids=prompt_ids,
        patch_images=patch_images,
        decoder_input_ids=decoder_input_ids,
    )
    # verified locally: OFAModel.forward() always returns Seq2SeqLMOutput with .logits
    # already projected to vocab size (the docstring calling it "hidden states" is stale/wrong)
    loss = loss_fn(output.logits.reshape(-1, output.logits.size(-1)), targets.reshape(-1))
    return loss


@torch.no_grad()
def evaluate_val_loss():
    model.eval()
    total_loss, n_batches = 0.0, 0
    for batch in val_loader:
        loss = run_forward(*batch)
        total_loss += loss.item()
        n_batches += 1
    model.train()
    return total_loss / max(n_batches, 1)


## 6. Train

A handful of epochs is expected to be enough given the dataset size (~2,069 examples). Watch `val_loss` for overfitting -- stop early if it starts rising.

In [ ]:
from tqdm.auto import tqdm

NUM_EPOCHS = 6
best_val_loss = float("inf")
os.makedirs("checkpoints/iu-xray-rebalanced", exist_ok=True)

for epoch in range(NUM_EPOCHS):
    epoch_loss, n_batches = 0.0, 0
    for batch in tqdm(train_loader, desc=f"epoch {epoch+1}/{NUM_EPOCHS}"):
        optimizer.zero_grad()
        loss = run_forward(*batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    train_loss = epoch_loss / n_batches
    val_loss = evaluate_val_loss()
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained("checkpoints/iu-xray-rebalanced")
        tokenizer.save_pretrained("checkpoints/iu-xray-rebalanced")
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")


## 7. Check whether mode collapse actually improved

Runs the same duplicate-rate / negation-aware keyword-F1 metric from the eval notebook
on the validation set, comparing this new checkpoint against the original
`iu-xray-finetuned` collapse rate (~60% duplicate rate, established across n=15/50/100).

In [ ]:
from transformers.generation_logits_process import LogitsProcessor, LogitsProcessorList

TEXT_VOCAB_SIZE = 50265

class RestrictToTextVocab(LogitsProcessor):
    def __init__(self, text_vocab_size):
        self.text_vocab_size = text_vocab_size

    def __call__(self, input_ids, scores):
        scores[:, self.text_vocab_size:] = -float("inf")
        return scores


@torch.no_grad()
def generate_description(model, image_path):
    image = Image.open(image_path)
    patch_image = transform(image).unsqueeze(0).to(DEVICE)
    prompt_ids = PROMPT_IDS.unsqueeze(0).to(DEVICE)
    logits_processor = LogitsProcessorList([RestrictToTextVocab(TEXT_VOCAB_SIZE)])
    generated_tokens = model.generate(
        prompt_ids, patch_images=patch_image, logits_processor=logits_processor,
        num_beams=5, no_repeat_ngram_size=3, max_length=64,
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0].strip()


model.eval()
results = []
for r in tqdm(val_dataset.records[:100], desc="generating on val set"):
    img_path = resolve_image_path(r["images"][0])
    description = generate_description(model, img_path)
    results.append({"generated_description": description, "ground_truth_report": r["response"]})

from collections import Counter
texts = [r["generated_description"] for r in results]
dup_counts = Counter(texts)
_, most_common_count = dup_counts.most_common(1)[0]
print(f"exact_duplicate_rate: {most_common_count/len(texts):.3f} (vs ~0.60 for the original iu-xray-finetuned checkpoint)")
print(f"unique_outputs: {len(dup_counts)}/{len(texts)}")
print()
print("Most common output:", dup_counts.most_common(1)[0])
print()
for r in results[:5]:
    print("GEN:  ", r["generated_description"])
    print("TRUTH:", r["ground_truth_report"][:150])
    print()


## 8. Save for download

The fine-tuned checkpoint is in `checkpoints/iu-xray-rebalanced/` in the same HF format
used throughout this project -- drop it into the local project's `checkpoints/` folder
and it's a direct drop-in replacement for `BiomedGPTCaptioner`, no conversion needed.

In [ ]:
import shutil

shutil.make_archive("iu-xray-rebalanced", "zip", "checkpoints/iu-xray-rebalanced")
from google.colab import files
files.download("iu-xray-rebalanced.zip")
